In [1]:
from vllm import LLM, SamplingParams

In [2]:
LLM_MODEL = "CohereForAI/aya-expanse-8b"  # you can use a better model
# LLM_MODEL = "ALLaM-AI/ALLaM-7B-Instruct-preview"  # you can use a better model
NUM_GPUs = 1

In [3]:
import torch 
torch.cuda.set_device(0)
torch.cuda.empty_cache()

In [4]:
llm = LLM(model=LLM_MODEL, tensor_parallel_size=NUM_GPUs, dtype="half")

INFO 03-05 06:11:15 __init__.py:207] Automatically detected platform cuda.


INFO 03-05 06:11:23 config.py:549] This model supports multiple tasks: {'score', 'classify', 'reward', 'generate', 'embed'}. Defaulting to 'generate'.
INFO 03-05 06:11:23 llm_engine.py:234] Initializing a V0 LLM engine (v0.7.3) with config: model='CohereForAI/aya-expanse-8b', speculative_config=None, tokenizer='CohereForAI/aya-expanse-8b', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.float16, max_seq_len=8192, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='xgrammar'), observability_config=ObservabilityConfig(otlp_traces_endpoint=None, collect_model_forward_time=False, collect_model_execute_time=False), seed=0, served_model_name=CohereForAI/aya-expanse-8b, num_s

tokenizer_config.json:   0%|          | 0.00/8.64k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/439 [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/21.0k [00:00<?, ?B/s]

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


INFO 03-05 06:16:19 model_runner.py:1115] Loading model weights took 14.9554 GB
INFO 03-05 06:16:22 worker.py:267] Memory profiling takes 3.36 seconds
INFO 03-05 06:16:22 worker.py:267] the current vLLM instance can use total_gpu_memory (23.64GiB) x gpu_memory_utilization (0.90) = 21.27GiB
INFO 03-05 06:16:22 worker.py:267] model weights take 14.96GiB; non_torch_memory takes 0.06GiB; PyTorch activation peak memory takes 2.39GiB; the rest of the memory reserved for KV Cache is 3.87GiB.
INFO 03-05 06:16:22 executor_base.py:111] # cuda blocks: 1982, # CPU blocks: 2048
INFO 03-05 06:16:22 executor_base.py:116] Maximum concurrency for 8192 tokens per request: 3.87x
INFO 03-05 06:16:24 model_runner.py:1434] Capturing cudagraphs for decoding. This may lead to unexpected consequences if the model is not static. To run the model in eager mode, set 'enforce_eager=True' or use '--enforce-eager' in the CLI. If out-of-memory error occurs during cudagraph capture, consider decreasing `gpu_memory_uti

Capturing CUDA graph shapes: 100%|██████████| 35/35 [00:20<00:00,  1.75it/s]

INFO 03-05 06:16:44 model_runner.py:1562] Graph capturing finished in 20 secs, took 0.25 GiB
INFO 03-05 06:16:44 llm_engine.py:436] init engine (profile, create kv cache, warmup model) took 25.52 seconds


In [5]:
def create_json_prompt_for_synthetic_data(**kwargs):
    
    # Use dictionary comprehension to filter out 'n/a' values and to keep the code flexible
    attributes = {key: value for key, value in kwargs.items() if value != "n/a"}
    
    # Building the initial part of the prompt
    prompt = """
**Objective:**
Produce realistic text passages in arabic that include clearly identified named entities. Each entity should be meticulously labeled according to its type for straightforward extraction.

**Format Requirements:**
- The output should be formatted in JSON, containing the text and the corresponding entities list.
- Each entity in the text should be accurately marked and annotated in the 'entities' list.
- Meticulously follow all the listed attributes.

**Entity Annotation Details:**
- Entity types can be multiwords separate by space. For instance, use "entity type" rather than "entity_type".
- Entities spans can be nested within other entities.
- A single entity may be associated with multiple types. list them in the key "types".

**Output Schema:**

<start attribute_1="value1" attribute_2="value2" ...>
{
  "text": "{text content}",
  "entities": [
    {"entity": "entity name", "types": ["type 1", "type 2", ...]},
    ...
  ]
}
<end>

**Here are some real world examples**:"""

    # Create a string of attributes for the <start> tag, excluding any 'n/a' values
    attributes_string = " ".join([f'{key}="{value}"' for key, value in attributes.items()])

    # Adding the dynamically created attributes string to the prompt
    prompt += f"""
<start {attributes_string}>
"""

    return prompt

In [6]:
sampling_params = SamplingParams(top_k=100, max_tokens=2000, top_p=0.8, stop="<end>")

In [7]:
import json

def generate(**kwargs):
    outputs = llm.generate([create_json_prompt_for_synthetic_data(**kwargs)], sampling_params)
    return json.loads(outputs[0].outputs[0].text)

In [8]:
generate(language="arabic", types_of_text="detailled job ads", sector="machine learning", country="egypt")


Processed prompts: 100%|██████████| 1/1 [00:04<00:00,  4.25s/it, est. speed input: 63.33 toks/s, output: 33.19 toks/s]


{'text': 'تُبحث شركة {company_name}، وهي شركة ناشئة رائدة في مجال الذكاء الاصطناعي في {egypt}، عن مهندس {job_title} لديه خبرة في {machine_learning_task}.',
 'entities': [{'entity': '{company_name}', 'types': ['company']},
  {'entity': '{egypt}', 'types': ['country']},
  {'entity': '{job_title}', 'types': ['job_title']},
  {'entity': '{machine_learning_task}', 'types': ['machine_learning_task']}]}

In [9]:
# post processing functions

import re

def tokenize_text(text):
    """Tokenize the input text into a list of tokens."""
    return re.findall(r'\w+(?:[-_]\w+)*|\S', text)

def extract_entities(data):
    all_examples = []

    for dt in data:

        # Attempt to extract entities; skip current record on failure
        try:
            tokens = tokenize_text(dt['text'])
            ents = [(k["entity"], k["types"]) for k in dt['entities']]
        except:
            continue

        spans = []
        for entity in ents:
            entity_tokens = tokenize_text(str(entity[0]))

            # Find the start and end indices of each entity in the tokenized text
            for i in range(len(tokens) - len(entity_tokens) + 1):
                if " ".join(tokens[i:i + len(entity_tokens)]).lower() == " ".join(entity_tokens).lower():
                    for el in entity[1]:
                        spans.append((i, i + len(entity_tokens) - 1, el.lower().replace('_', ' ')))

        # Append the tokenized text and its corresponding named entity recognition data
        all_examples.append({"tokenized_text": tokens, "ner": spans})

    return all_examples

# generation functions
def generate_from_prompts(prompts, llm, sampling_params):
    outputs = llm.generate(prompts, sampling_params)

    all_outs = []
    
    for output in outputs:
        try:
            js = json.loads(output.outputs[0].text.strip())
        except:
            continue
            
        all_outs.append(js)

    return all_outs, extract_entities(all_outs)

In [10]:
countries = [
    "مدغشقر", "تايوان", "الولايات المتحدة", "ألمانيا", "فرنسا", "إسبانيا", "روسيا", "الصين", 
    "اليابان", "البرازيل", "الهند", "مصر", "جنوب إفريقيا", "أستراليا", "كندا", 
    "المكسيك", "إندونيسيا", "نيجيريا", "تركيا", "المملكة المتحدة", "إيطاليا", "بولندا", 
    "الأرجنتين", "هولندا", "بلجيكا", "سويسرا", "السويد", "النرويج", "فنلندا",
    "الدنمارك", "البرتغال", "اليونان", "إيران", "تايلاند", "الفلبين", "فيتنام", 
    "كوريا الجنوبية", "المملكة العربية السعودية", "إسرائيل", "الإمارات العربية المتحدة", 
    "نيوزيلندا", "أيرلندا", "ماليزيا", "سنغافورة", "هونغ كونغ", "جمهورية التشيك", 
    "المجر", "رومانيا", "كولومبيا", "بيرو", "فنزويلا", "تشيلي", "المغرب", "الجزائر", 
    "تونس", "نيبال", "باكستان", "بنغلاديش", "كازاخستان", "أوكرانيا", "النمسا", 
    "كرواتيا", "صربيا", "كينيا", "غانا", "زيمبابوي", "كوبا", "بنما", "فيجي", 
    "منغوليا", "كوريا الشمالية", "ميانمار", "إثيوبيا", "تنزانيا", "ليبيا", 
    "الأردن", "قطر", "عُمان", "الكويت", "لبنان", "بلغاريا", "سلوفاكيا", "ليتوانيا", 
    "لاتفيا", "إستونيا", "قبرص", "لوكسمبورغ", "ماكاو", "بوتان", "المالديف", 
    "أنغولا", "الكاميرون", "السنغال", "مالي", "زامبيا", "أوغندا", "ناميبيا", 
    "بوتسوانا", "موزمبيق", "ساحل العاج", "بوركينا فاسو", "مالاوي", "الغابون", 
    "ليسوتو", "غامبيا", "غينيا", "الرأس الأخضر", "رواندا", "بنين", "بوروندي", 
    "الصومال", "إريتريا", "جيبوتي", "توغو", "سيشيل", "تشاد", "جمهورية إفريقيا الوسطى", 
    "ليبيريا", "موريتانيا", "سريلانكا", "سيراليون", "غينيا الاستوائية", "إسواتيني", 
    "الكونغو (كينشاسا)", "الكونغو (برازافيل)",
    # Added culturally relevant countries
    "السودان", "اليمن", "البحرين", "فلسطين", "سوريا", "العراق", "جزر القمر", 
    "مالطا"  # Historical trade ties with Arab world
]

In [11]:
job_sectors = [
    # تخصصات القطاع المالي (Finance Sector Specializations)
    "الخدمات المصرفية الاستثمارية",
    "التمويل المؤسسي",
    "إدارة الأصول",
    "إدارة المخاطر",
    "التحليل الكمي",
    "التخطيط المالي",
    "التمويل الإسلامي",  # Added: Islamic Finance, prominent in Arab world
    "إدارة الزكاة والأوقاف",  # Added: Zakat and Waqf management

    # تخصصات التعلم الآلي والذكاء الاصطناعي (Machine Learning and AI Specializations)
    "معالجة اللغة الطبيعية",
    "رؤية الحاسوب",
    "التعلم العميق",
    "التعلم المعزز",
    "التحليلات التنبؤية",
    "تطوير الخوارزميات",
    "ترجمة اللغة العربية الآلية",  # Added: Arabic language translation tech

    # تخصصات القطاع الصحي (Healthcare Sector Specializations)
    "البحوث الطبية",
    "التجارب السريرية",
    "معلوماتية الصحة",
    "الهندسة الطبية الحيوية",
    "إدارة الصحة العامة",
    "الصناعات الدوائية",
    "الطب التقليدي العربي",  # Added: Traditional Arab medicine (e.g., herbal remedies)

    # تخصصات القطاع التعليمي (Education Sector Specializations)
    "تطوير المناهج",
    "تكنولوجيا التعليم",
    "التعليم الخاص",
    "إدارة التعليم العالي",
    "سياسة التعليم",
    "تعليم اللغة",
    "تدريس اللغة العربية والدراسات الإسلامية",  # Added: Arabic and Islamic studies

    # تخصصات القطاع الصناعي (Manufacturing Sector Specializations)
    "هندسة العمليات",
    "مراقبة الجودة",
    "التصميم الصناعي",
    "تحسين سلسلة التوريد",
    "تصنيع الروبوتات",
    "التصنيع الخالي من الهدر",
    "صناعة النسيج التقليدي",  # Added: Traditional textile industry

    # تخصصات قطاع الطاقة (Energy Sector Specializations)
    "أنظمة الطاقة المتجددة",
    "استكشاف النفط والغاز",
    "استشارات كفاءة الطاقة",
    "الهندسة النووية",
    "تكنولوجيا الشبكات الذكية",
    "سياسة الطاقة",
    "إدارة الطاقة الصحراوية",  # Added: Desert energy management (e.g., solar in deserts)

    # تخصصات القطاع البيئي (Environmental Sector Specializations)
    "حماية الحياة البرية",
    "علوم البيئة",
    "إدارة الموارد المائية",
    "استراتيجية الاستدامة",
    "تحليل تغير المناخ",
    "القانون البيئي",
    "الزراعة الصحراوية",  # Added: Desert agriculture, key in Arab regions

    # تخصصات الإعلام والاتصالات (Media and Communications Specializations)
    "التسويق الرقمي",
    "الصحافة",
    "العلاقات العامة",
    "إنتاج الأفلام",
    "البث الإذاعي",
    "استراتيجية المحتوى",
    "إنتاج المحتوى الثقافي العربي",  # Added: Arabic cultural content production

    # تخصصات القطاع القانوني (Legal Sector Specializations)
    "القانون التجاري",
    "القانون الدولي",
    "الملكية الفكرية",
    "القانون البيئي",
    "التقاضي المدني",
    "الدفاع الجنائي",
    "الشريعة والقانون الإسلامي",  # Added: Sharia and Islamic law

    # تخصصات قطاع التجزئة (Retail Sector Specializations)
    "استراتيجية التجارة الإلكترونية",
    "إدارة المتاجر",
    "تخطيط البضائع",
    "إدارة تجربة العملاء",
    "تحليلات التجزئة",
    "لوجستيات سلسلة التوريد",
    "تجارة الأسواق التقليدية",  # Added: Traditional market trade (e.g., souks)
]

In [12]:
import random

countries = ["السعودية", "مصر", "الإمارات", "المغرب", "لبنان", "الجزائر", "تونس", "الأردن", "قطر", "الكويت"] # Example countries
job_sectors = ["مبرمج", "مهندس", "طبيب", "محاسب", "مسوق", "مصمم", "مدير مشروع", "مدرس", "صحفي", "محامي"] # Example job sectors

domains = {
    "إعلانات الوظائف": {  # Job Ads
        "entities": ["اسم الشركة", "المسمى الوظيفي", "الموقع", "الراتب", "المهارات المطلوبة"],
        "examples": {
            "اسم الشركة": [f"{c} للتقنية" for c in countries],  # Uses countries list
            "المسمى الوظيفي": job_sectors,
            "الموقع": countries,
            "الراتب": [f"{random.randint(5000, 30000)} ريال سعودي" for _ in range(10)],  # Pre-generate some salary examples
            "المهارات المطلوبة": job_sectors
        }
    },
    "الطب العربي": {  # Arabic Medicine
        "entities": ["اسم الطبيب", "العلاج التقليدي", "المكونات", "الحالة المرضية", "الموقع"],
        "examples": {
            "اسم الطبيب": ["الدكتور أحمد النجار", "الحكيم خالد بن يوسف", "المعالجة فاطمة الزهراء"],
            "العلاج التقليدي": ["الكي بالنار", "الحجامة", "الأعشاب الطبية"],
            "المكونات": ["العسل", "حبة البركة", "زيت الزيتون", "الزنجبيل"],
            "الحالة المرضية": ["الصداع", "الحمى", "آلام المفاصل"],
            "الموقع": countries
        }
    },
    "الدين الإسلامي": {  # Arabic Religion (Islamic Context)
        "entities": ["اسم العالم", "الموضوع الديني", "المكان", "الحدث", "النص الديني"],
        "examples": {
            "اسم العالم": ["الشيخ محمد بن صالح", "الإمام عبد الرحمن الدوسري", "الداعية نورة السعدي"],
            "الموضوع الديني": ["تفسير القرآن", "الصلاة", "الصيام", "الحج"],
            "المكان": ["المسجد الحرام", "المسجد النبوي", "مسجد القدس"] + countries,
            "الحدث": ["درس ديني", "خطبة الجمعة", "محاضرة رمضانية"],
            "النص الديني": ["سورة الفاتحة", "حديث البخاري", "آية الكرسي"]
        }
    },
    "الرياضة": {  # Sports
        "entities": ["اسم الفريق", "نوع الرياضة", "الموقع", "الحدث الرياضي", "اللاعب المميز"],
        "examples": {
            "اسم الفريق": ["نادي الهلال", "نادي النصر", "فريق الاتحاد"] + [f"نادي {c}" for c in countries[:10]],  # Fixed: Uses countries list, limited to 10 for brevity
            "نوع الرياضة": ["كرة القدم", "سباق الهجن", "الفروسية", "كرة السلة"],
            "الموقع": countries,
            "الحدث الرياضي": ["مباراة نهائية", "بطولة محلية", "سباق صحراوي"],
            "اللاعب المميز": ["محمد صلاح", "عبد الرزاق حمدالله", "يوسف الخليفي"]
        }
    },
    "المطبخ العربي": {  # Arabic Cuisine
        "entities": ["اسم الطبق", "المكونات الرئيسية", "نوع المطبخ", "طريقة التحضير", "المناسبة"],
        "examples": {
            "اسم الطبق": ["الكبسة", "المندي", "المقلوبة", "الكشري", "الطاجين"],
            "المكونات الرئيسية": ["الأرز", "اللحم", "الدجاج", "الخضروات", "البهارات"],
            "نوع المطبخ": ["خليجي", "مصري", "شامي", "مغربي", "يمني"],
            "طريقة التحضير": ["مشوي", "مطهو", "مقلي", "مبخر", "محشي"],
            "المناسبة": ["عيد", "رمضان", "عشاء عائلي", "غداء يومي", "احتفال خاص"]
        }
    },
    "الأدب والشعر العربي": {  # Arabic Literature and Poetry
        "entities": ["اسم الشاعر/الأديب", "عنوان القصيدة/الرواية", "نوع الأدب", "الموضوع", "الفترة الزمنية"],
        "examples": {
            "اسم الشاعر/الأديب": ["أحمد شوقي", "نجيب محفوظ", "جبران خليل جبران", "محمود درويش", "أحلام مستغانمي"],
            "عنوان القصيدة/الرواية": ["النهج البردة", "أولاد حارتنا", "النبي", "مديح الظل العالي", "ذاكرة الجسد"],
            "نوع الأدب": ["شعر عمودي", "شعر حر", "رواية", "قصة قصيرة", "مقال"],
            "الموضوع": ["الحب", "الوطن", "الفراق", "الحكمة", "الطبيعة"],
            "الفترة الزمنية": ["العصر الجاهلي", "العصر العباسي", "العصر الحديث", "الأندلس", "القرن العشرين"]
        }
    },
    "التاريخ والأحداث التاريخية": {  # History and Historical Events
        "entities": ["اسم الشخصية التاريخية", "الحدث التاريخي", "المكان التاريخي", "الفترة الزمنية", "الدولة/الحضارة"],
        "examples": {
            "اسم الشخصية التاريخية": ["صلاح الدين الأيوبي", "عمر بن الخطاب", "الملك عبد العزيز", "جميلة بوحيرد", "طه حسين"],
            "الحدث التاريخي": ["فتح القدس", "معركة القادسية", "تأسيس المملكة العربية السعودية", "ثورة الجزائر", "حرب 1967"],
            "المكان التاريخي": ["القدس", "دمشق", "القاهرة", "مكة", "الأندلس"],
            "الفترة الزمنية": ["العصر الأموي", "العصر العباسي", "الحرب العالمية الثانية", "الاستعمار", "العصور الوسطى"],
            "الدولة/الحضارة": ["الدولة الأموية", "الدولة العباسية", "الدولة العثمانية", "الحضارة الإسلامية", "مصر القديمة"]
        }
    },
    "الموسيقى والفنون العربية": {  # Arabic Music and Arts
        "entities": ["اسم الفنان/الموسيقي", "نوع الفن/الموسيقى", "الآلة الموسيقية", "العمل الفني", "الأسلوب الفني"],
        "examples": {
            "اسم الفنان/الموسيقي": ["أم كلثوم", "فيروز", "محمد عبده", "كاظم الساهر", "نجوى كرم"],
            "نوع الفن/الموسيقى": ["موسيقى كلاسيكية", "موسيقى شعبية", "أغنية طربية", "فن الخط العربي", "فن النحت"],
            "الآلة الموسيقية": ["العود", "القانون", "الناي", "الكمان", "الطبلة"],
            "العمل الفني": ["أنت عمري", "يا جارة الوادي", "الأطلال", "لوحة خط", "تمثال"],
            "الأسلوب الفني": ["طرب أصيل", "موسيقى حديثة", "خط كوفي", "نحت إسلامي", "رسم تجريدي"]
        }
    },
    "التكنولوجيا والابتكار": {  # Technology and Innovation
        "entities": ["اسم الشركة التقنية", "المنتج/الخدمة التقنية", "المجال التقني", "المخترع/المطور", "الموقع التقني (مدينة/دولة)"],
        "examples": {
            "اسم الشركة التقنية": ["أرامكو تك", "اتصالات", "زين", "جاهز", "نون"],
            "المنتج/الخدمة التقنية": ["تطبيق توصيل", "منصة تجارة إلكترونية", "نظام سحابي", "برنامج ذكاء اصطناعي", "شبكة اتصالات 5G"],
            "المجال التقني": ["الذكاء الاصطناعي", "الأمن السيبراني", "الحوسبة السحابية", "إنترنت الأشياء", "التكنولوجيا المالية"],
            "المخترع/المطور": ["مهندس برمجيات", "عالم بيانات", "مطور تطبيقات", "باحث في الذكاء الاصطناعي", "فريق هندسي"],
            "الموقع التقني (مدينة/دولة)": ["مدينة الملك عبدالله الاقتصادية", "دبي انترنت سيتي", "وادي السيليكون", "القاهرة الذكية", "الرياض التقنية"]
        }
    },
    "السياحة والسفر": {  # Travel and Tourism
        "entities": ["اسم الوجهة السياحية", "نوع السياحة", "وسيلة السفر", "النشاط السياحي", "الموسم السياحي"],
        "examples": {
            "اسم الوجهة السياحية": ["الأهرامات", "المسجد النبوي", "برج خليفة", "وادي رم", "شواطئ البحر الأحمر"],
            "نوع السياحة": ["سياحة دينية", "سياحة ثقافية", "سياحة ترفيهية", "سياحة مغامرات", "سياحة علاجية"],
            "وسيلة السفر": ["طائرة", "سيارة", "قطار", "سفينة سياحية", "حافلة"],
            "النشاط السياحي": ["زيارة المواقع الأثرية", "التسوق في الأسواق التقليدية", "الغطس والغوص", "رحلات السفاري", "الاسترخاء في المنتجعات"],
            "الموسم السياحي": ["الصيف", "الشتاء", "رمضان", "عيد الأضحى", "مهرجان الجنادرية"]
        }
    },
    "الألعاب والترفيه": {  # Games and Entertainment
        "entities": ["اسم اللعبة", "نوع اللعبة", "المنصة", "المطور", "الشخصية الرئيسية"],
        "examples": {
            "اسم اللعبة": ["ببجي موبايل", "فري فاير", "فورتنايت", "كول أوف ديوتي", "فيفا"],
            "نوع اللعبة": ["ألعاب قتالية", "ألعاب استراتيجية", "ألعاب رياضية", "ألعاب تقمص الأدوار", "ألعاب ألغاز"],
            "المنصة": ["الهواتف الذكية", "الحاسوب الشخصي", "بلايستيشن", "إكس بوكس", "نينتندو سويتش"],
            "المطور": ["تينسنت جيمز", "جارينا", "إيبك جيمز", "أكتيفيجن", "إي أيه سبورتس"],
            "الشخصية الرئيسية": ["شخصية ببجي", "شخصية فري فاير", "شخصية فورتنايت", "الكابتن برايس", "لاعب كرة قدم"]
        }
    },
    "الأفلام والسينما": {  # Movies and Cinema
        "entities": ["اسم الفيلم", "نوع الفيلم", "المخرج", "الممثل الرئيسي", "سنة الإنتاج"],
        "examples": {
            "اسم الفيلم": ["الرسالة", "عمر المختار", "باب الشمس", "وجدة", "كفرناحوم"],
            "نوع الفيلم": ["فيلم تاريخي", "فيلم درامي", "فيلم كوميدي", "فيلم أكشن", "فيلم رومانسي"],
            "المخرج": ["مصطفى العقاد", "أنطوني كوين", "يسري نصر الله", "هيفاء المنصور", "نادين لبكي"],
            "الممثل الرئيسي": ["عبدالله غيث", "أنتوني كوين", "هند صبري", "وعد محمد", "زين الرافعي"],
            "سنة الإنتاج": ["1976", "1981", "2004", "2012", "2018"]
        }
    },
    "التسوق والأزياء": {  # Shopping and Fashion
        "entities": ["اسم المنتج", "نوع الملابس", "الماركة", "المتجر", "المناسبة"],
        "examples": {
            "اسم المنتج": ["فستان سهرة", "قميص رجالي", "حذاء رياضي", "حقيبة يد", "عباءة"],
            "نوع الملابس": ["ملابس نسائية", "ملابس رجالية", "ملابس أطفال", "ملابس رياضية", "ملابس تقليدية"],
            "الماركة": ["زارا", "أديداس", "نايكي", "شانيل", "ديور"],
            "المتجر": ["مول الإمارات", "رد تاغ", "نمشي", "سوق.كوم", "محلات التجزئة"],
            "المناسبة": ["حفل زفاف", "اجتماع عمل", "يوم عادي", "رياضة", "عيد"]
        }
    },
    "السيارات والمركبات": {  # Cars and Vehicles
        "entities": ["اسم السيارة", "نوع السيارة", "الماركة", "الموديل", "المواصفات"],
        "examples": {
            "اسم السيارة": ["تويوتا كامري", "مرسيدس بنز سي كلاس", "فورد موستانج", "نيسان باترول", "هيونداي سوناتا"],
            "نوع السيارة": ["سيدان", "SUV", "بيك أب", "رياضية", "فاخرة"],
            "الماركة": ["تويوتا", "مرسيدس بنز", "فورد", "نيسان", "هيونداي"],
            "الموديل": ["كامري", "سي كلاس", "موستانج", "باترول", "سوناتا"],
            "المواصفات": ["محرك قوي", "دفع رباعي", "تصميم رياضي", "اقتصادية في الوقود", "مريحة"]
        }
    },
    "الطقس والمناخ": {  # Weather and Climate
        "entities": ["المدينة", "درجة الحرارة", "حالة الطقس", "الرياح", "الرطوبة"],
        "examples": {
            "المدينة": countries + ["الرياض", "جدة", "القاهرة", "دبي", "بيروت"],
            "درجة الحرارة": [f"{random.randint(10, 45)} درجة مئوية" for _ in range(10)],  # Pre-generate temperature examples
            "حالة الطقس": ["مشمس", "غائم جزئي", "ممطر", "عاصف", "ضبابي"],
            "الرياح": ["خفيفة", "معتدلة", "قوية", "شمالية", "جنوبية"],
            "الرطوبة": [f"{random.randint(20, 90)}%" for _ in range(10)]  # Pre-generate humidity examples
        }
    },
    "الفعاليات والمناسبات": {  # Events and Occasions
        "entities": ["اسم الفعالية", "نوع الفعالية", "الموقع", "التاريخ", "الجمهور المستهدف"],
        "examples": {
            "اسم الفعالية": ["مهرجان الجنادرية", "معرض الكتاب الدولي", "موسم الرياض", "سباق الفورمولا 1", "احتفالات اليوم الوطني"],
            "نوع الفعالية": ["مهرجان ثقافي", "معرض تجاري", "فعالية ترفيهية", "حدث رياضي", "احتفال وطني"],
            "الموقع": countries + ["مركز المعارض", "الاستاد الرياضي", "المنطقة التاريخية", "الساحة العامة", "المسرح"],
            "التاريخ": ["نهاية الأسبوع", "شهر رمضان", "فصل الشتاء", "يوم العيد", "العطلة الصيفية"],
            "الجمهور المستهدف": ["العائلات", "الشباب", "رجال الأعمال", "السياح", "المثقفون"]
        }
    },
    "وسائل التواصل الاجتماعي": {  # Social Media
        "entities": ["اسم المنصة", "المستخدم", "نوع المحتوى", "الموضوع الشائع (الهاشتاج)", "التفاعل (إعجابات/تعليقات/مشاركات)"],
        "examples": {
            "اسم المنصة": ["تويتر", "انستغرام", "فيسبوك", "يوتيوب", "تيك توك"],
            "المستخدم": ["شخصية مشهورة", "مدون", "صفحة إخبارية", "شركة", "حساب شخصي"],
            "نوع المحتوى": ["تغريدة", "صورة", "فيديو قصير", "منشور", "قصة"],
            "الموضوع الشائع (الهاشتاج)": ["#موسم_الرياض", "#رمضان_كريم", "#اليوم_الوطني", "#العودة_للمدارس", "#كأس_العالم"],
            "التفاعل (إعجابات/تعليقات/مشاركات)": [f"{random.randint(100, 10000)} إعجاب" for _ in range(5)] + 
                                                    [f"{random.randint(10, 500)} تعليق" for _ in range(5)] + 
                                                    [f"{random.randint(50, 2000)} مشاركة" for _ in range(5)]  # Mixed examples
        }
    },
    "التعليم والمدارس": {  # Education and Schools
        "entities": ["اسم المدرسة/الجامعة", "المرحلة التعليمية", "المادة الدراسية", "المعلم/الأستاذ", "النشاط المدرسي"],
        "examples": {
            "اسم المدرسة/الجامعة": ["جامعة الملك سعود", "جامعة القاهرة", "مدرسة الرواد", "جامعة بيروت العربية", "مدرسة الأندلس"],
            "المرحلة التعليمية": ["ابتدائي", "متوسط", "ثانوي", "جامعي", "دراسات عليا"],
            "المادة الدراسية": ["الرياضيات", "العلوم", "اللغة العربية", "التاريخ", "الجغرافيا"],
            "المعلم/الأستاذ": ["الأستاذ أحمد", "المعلمة فاطمة", "الدكتور خالد", "الشيخة نورة", "البروفيسور علي"],
            "النشاط المدرسي": ["يوم مفتوح", "رحلة ميدانية", "مسابقة ثقافية", "معرض فني", "درس نموذجي"]
        }
    },
    "الكتب والمكتبات": {  # Books and Libraries
        "entities": ["اسم الكتاب", "المؤلف", "نوع الكتاب", "دار النشر", "الموضوع الرئيسي"],
        "examples": {
            "اسم الكتاب": ["ألف ليلة وليلة", "العادات السبع للناس الأكثر فعالية", "رجال من المريخ ونساء من الزهرة", "مقدمة ابن خلدون", "لا تحزن"],
            "المؤلف": ["مجهول", "ستيفن كوفي", "جون غراي", "ابن خلدون", "عائض القرني"],
            "نوع الكتاب": ["أدب", "تطوير الذات", "علم نفس", "تاريخ", "دين"],
            "دار النشر": ["دار الشروق", "مكتبة العبيكان", "الدار العربية للعلوم ناشرون", "مؤسسة هنداوي", "مركز الأدب العربي"],
            "الموضوع الرئيسي": ["قصص خيالية", "الفعالية الشخصية", "العلاقات بين الجنسين", "التاريخ الإسلامي", "التفاؤل"]
        }
    }
}

## Generate Prompts

In [30]:
# create prompts
NUM_SAMPLES = 100_000

import random

all_prompts = []

for i in range(NUM_SAMPLES):
    # Randomly select a domain
    domain = random.choice(list(domains.keys()))
    entities = domains[domain]["entities"]
    examples = domains[domain]["examples"]

    kwargs = {"language": "arabic"}
    for entity in entities:
        kwargs[entity] = random.choice(examples[entity])

    # Generate prompt
    prompt = create_json_prompt_for_synthetic_data(domain=domain, entities=", ".join(entities), **kwargs)
    all_prompts.append(prompt)

In [31]:
output, processed_output = generate_from_prompts(all_prompts, llm, sampling_params)


Processed prompts:   0%|          | 0/100000 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

WARNING 03-05 06:24:43 scheduler.py:1754] Sequence group 104 is preempted by PreemptionMode.RECOMPUTE mode because there is not enough KV cache space. This can affect the end-to-end performance. Increase gpu_memory_utilization or tensor_parallel_size to provide more KV cache memory. total_num_cumulative_preemption=1


Processed prompts:   0%|          | 210/100000 [01:02<4:39:10,  5.96it/s, est. speed input: 1095.92 toks/s, output: 625.36 toks/s] 

WARNING 03-05 06:25:41 scheduler.py:1754] Sequence group 295 is preempted by PreemptionMode.RECOMPUTE mode because there is not enough KV cache space. This can affect the end-to-end performance. Increase gpu_memory_utilization or tensor_parallel_size to provide more KV cache memory. total_num_cumulative_preemption=51


Processed prompts:   1%|          | 545/100000 [02:28<6:56:39,  3.98it/s, est. speed input: 1200.18 toks/s, output: 684.09 toks/s] 

...

Processed prompts: 100%|█████████▉| 99511/100000 [7:12:53<01:57,  4.16it/s, est. speed input: 1253.70 toks/s, output: 718.88 toks/s]

WARNING 03-05 13:37:32 scheduler.py:1754] Sequence group 99590 is preempted by PreemptionMode.RECOMPUTE mode because there is not enough KV cache space. This can affect the end-to-end performance. Increase gpu_memory_utilization or tensor_parallel_size to provide more KV cache memory. total_num_cumulative_preemption=14501


Processed prompts: 100%|█████████▉| 99882/100000 [7:14:29<00:25,  4.61it/s, est. speed input: 1253.72 toks/s, output: 718.91 toks/s]

WARNING 03-05 13:39:08 scheduler.py:1754] Sequence group 99964 is preempted by PreemptionMode.RECOMPUTE mode because there is not enough KV cache space. This can affect the end-to-end performance. Increase gpu_memory_utilization or tensor_parallel_size to provide more KV cache memory. total_num_cumulative_preemption=14551


Processed prompts: 100%|██████████| 100000/100000 [7:14:53<00:00,  3.83it/s, est. speed input: 1254.06 toks/s, output: 719.16 toks/s]


In [32]:
output[1]

{'text': 'فيفا هي لعبة فيديو من تطوير إي أيه سبورتس. اللعبة متاحة على منصة إكس بوكس وتُعرف بشخصيتها الرئيسية المميزة، وهي شخصية من لعبة فورتنايت الشهيرة.',
 'entities': [{'entity': 'فيفا', 'types': ['اسم اللعبة', 'نوع اللعبة']},
  {'entity': 'إي أيه سبورتس', 'types': ['المطور']},
  {'entity': 'إكس بوكس', 'types': ['المنصة']},
  {'entity': 'فورتنايت', 'types': ['الشخصية الرئيسية']},
  {'entity': 'شخصية فورتنايت', 'types': ['الشخصية الرئيسية', 'نوع اللعبة']}]}

In [41]:
output


[{'text': "في كتابه المشهور 'رجال من المريخ ونساء من الزهرة'، يستكشف المؤلف ابن خلدون الطبيعة البشرية من خلال قصص خيالية، حيث يصور كيف يمكن للاختلافات بين الجنسين أن تشكل تفاعلاتنا وقراراتنا. يُنسب إلى هذا الكتاب الفضل في تقديم رؤى ثاقبة حول ديناميكيات العلاقات الإنسانية، وقد حظي بشعبية واسعة بين القراء المهتمين بالعلوم الاجتماعية.",
  'entities': [{'entity': 'اسم الكتاب',
    'types': ['كتاب'],
    'value': 'رجال من المريخ ونساء من الزهرة'},
   {'entity': 'المؤلف', 'types': ['شخص'], 'value': 'ابن خلدون'},
   {'entity': 'نوع الكتاب', 'types': ['نوع'], 'value': 'تاريخ'},
   {'entity': 'دار النشر',
    'types': ['منظمة'],
    'value': 'الدار العربية للعلوم ناشرون'},
   {'entity': 'الموضوع الرئيسي', 'types': ['موضوع'], 'value': 'قصص خيالية'}]},
 {'text': 'فيفا هي لعبة فيديو من تطوير إي أيه سبورتس. اللعبة متاحة على منصة إكس بوكس وتُعرف بشخصيتها الرئيسية المميزة، وهي شخصية من لعبة فورتنايت الشهيرة.',
  'entities': [{'entity': 'فيفا', 'types': ['اسم اللعبة', 'نوع اللعبة']},
   {'entity': 'إي

In [42]:
processed_output

[{'tokenized_text': ['في',
   'كتابه',
   'المشهور',
   "'",
   'رجال',
   'من',
   'المريخ',
   'ونساء',
   'من',
   'الزهرة',
   "'",
   '،',
   'يستكشف',
   'المؤلف',
   'ابن',
   'خلدون',
   'الطبيعة',
   'البشرية',
   'من',
   'خلال',
   'قصص',
   'خيالية',
   '،',
   'حيث',
   'يصور',
   'كيف',
   'يمكن',
   'للاختلافات',
   'بين',
   'الجنسين',
   'أن',
   'تشكل',
   'تفاعلاتنا',
   'وقراراتنا',
   '.',
   'ي',
   'ُ',
   'نسب',
   'إلى',
   'هذا',
   'الكتاب',
   'الفضل',
   'في',
   'تقديم',
   'رؤى',
   'ثاقبة',
   'حول',
   'ديناميكيات',
   'العلاقات',
   'الإنسانية',
   '،',
   'وقد',
   'حظي',
   'بشعبية',
   'واسعة',
   'بين',
   'القراء',
   'المهتمين',
   'بالعلوم',
   'الاجتماعية',
   '.'],
  'ner': [(13, 13, 'شخص')]},
 {'tokenized_text': ['فيفا',
   'هي',
   'لعبة',
   'فيديو',
   'من',
   'تطوير',
   'إي',
   'أيه',
   'سبورتس',
   '.',
   'اللعبة',
   'متاحة',
   'على',
   'منصة',
   'إكس',
   'بوكس',
   'وت',
   'ُ',
   'عرف',
   'بشخصيتها',
   'الرئيسية',
   'المم

In [43]:
lengths = []

for d in processed_output:
    lengths.append(len(d["tokenized_text"]))

print("Avg num tokens:", sum(lengths) / len(lengths))

Avg num tokens: 36.8026034650936


In [44]:
len_ner = []

for d in processed_output:
    len_ner.append(len(d["ner"]))
        
print("Avg num of entities:", sum(len_ner) / len(len_ner))

Avg num of entities: 3.964338798829486


In [45]:
unique_entities = []

for d in processed_output:
    for n in d["ner"]:
        unique_entities.append((str(n[2]).lower()))

print("Unique entity types:", len(unique_entities))

Unique entity types: 341393


In [48]:
Counter(unique_entities).most_common()[:100]

[('الموقع', 14670),
 ('الماركة', 9457),
 ('الفترة الزمنية', 7987),
 ('اسم الوجهة السياحية', 6818),
 ('اسم الطبق', 6776),
 ('اسم الفنان/الموسيقي', 6193),
 ('اسم الفيلم', 5885),
 ('اسم اللعبة', 5294),
 ('اسم الشخصية التاريخية', 5290),
 ('المكان التاريخي', 5286),
 ('الممثل الرئيسي', 5237),
 ('المادة الدراسية', 5146),
 ('اسم الفريق', 4996),
 ('عنوان القصيدة/الرواية', 4995),
 ('الموضوع الديني', 4923),
 ('اللاعب المميز', 4893),
 ('اسم العالم', 4811),
 ('سنة الإنتاج', 4792),
 ('اسم الشاعر/الأديب', 4791),
 ('الآلة الموسيقية', 4687),
 ('الموديل', 4659),
 ('المكونات الرئيسية', 4640),
 ('المعلم/الأستاذ', 4631),
 ('اسم السيارة', 4628),
 ('المطور', 4621),
 ('المخرج', 4584),
 ('اسم المدرسة/الجامعة', 4443),
 ('المكان', 4360),
 ('المنصة', 4342),
 ('نوع الفيلم', 4287),
 ('الموسم السياحي', 4284),
 ('الموضوع', 4174),
 ('الشخصية الرئيسية', 4146),
 ('الدولة/الحضارة', 4099),
 ('العمل الفني', 4064),
 ('التاريخ', 3905),
 ('الحدث الرياضي', 3850),
 ('اسم الشركة التقنية', 3846),
 ('المجال التقني', 3750),
 ('المو

In [49]:
def save_data_to_file(data, filepath):
    """Saves the processed data to a JSON file."""
    with open(filepath, 'w') as f:
        json.dump(data, f)

In [50]:
output_file = "100k_aya_expanse_gliner.json"

save_data_to_file(processed_output, output_file)